# Filesystem File Search Middleware

Middleware that provides file-path and file-content search tools for files stored on the local filesystem.

It exposes:

- `glob_search` — Finds files using glob patterns.
- `grep_search` — Searches file contents using regular expressions.

The middleware restricts every search to a configured root directory and prevents path traversal or access through symlinks that resolve outside that root.

---

# `FilesystemFileSearchMiddleware`

Agent middleware that adds filesystem-based Glob and Grep tools to an agent.

- Bases: `AgentMiddleware[AgentState[ResponseT], ContextT, ResponseT]`

The middleware can use the external `ripgrep` command for faster content searches and automatically falls back to a Python regex-based implementation when `ripgrep` is disabled, unavailable, or times out.

## Constructor

```python
FilesystemFileSearchMiddleware(
    *,
    root_path: str, # Root directory within which searches are allowed
    use_ripgrep: bool = True, # Whether to prefer ripgrep for content searches
    max_file_size_mb: int = 10 # Maximum file size searched by the Python fallback
) -> None
```

## Attributes

- `root_path` — Resolved `Path` representing the root directory within which all searches must remain.
- `use_ripgrep` — Determines whether `grep_search` first attempts to search using `ripgrep`.
- `max_file_size_bytes` — Maximum file size, in bytes, that the Python fallback is allowed to read.
- `glob_search` — Structured agent tool used to locate files by glob pattern.
- `grep_search` — Structured agent tool used to search file contents with a regular expression.
- `tools` — List containing `glob_search` and `grep_search` for registration with an agent.

---

## Tools

### 1. `glob_search`

Finds files whose paths match a supplied glob pattern.

It supports patterns such as `**/*.js` and `src/**/*.ts`. Matching paths are returned as virtual paths relative to `root_path`, beginning with `/`.

Results are sorted by file modification time, with the most recently modified files first.

- **Syntax:**

  ```python
  glob_search(
      pattern: str, # Glob pattern used to match files
      path: str = "/" # Virtual directory from which the search starts
  ) -> str
  ```

- `pattern` — Glob expression used to match file paths.
- `path` — Search directory represented as a virtual path relative to `root_path`.
- **Returns:** Newline-separated matching file paths.
- **No-result output:** `"No files found"`.

The tool rejects:

- Absolute glob patterns.
- Patterns containing a `..` path segment.
- Search paths that escape the configured root.
- Files reached through symlinks that resolve outside the configured root.

---

### 2. `grep_search`

Searches file contents using a regular expression.

The tool can optionally restrict the search to filenames matching an include pattern and supports three output formats.

- **Syntax:**

  ```python
  grep_search(
      pattern: str, # Regular expression searched for in file contents
      path: str = "/", # Virtual file or directory to search
      include: str | None = None, # Optional filename glob filter
      output_mode: Literal[
          "files_with_matches",
          "content",
          "count"
      ] = "files_with_matches"
  ) -> str
  ```

- `pattern` — Regular expression applied to file contents.
- `path` — Virtual file or directory path relative to `root_path`.
- `include` — Optional filename filter such as `"*.js"` or `"*.{ts,tsx}"`.
- `output_mode` — Controls how matching results are formatted.

#### Output modes

- `"files_with_matches"` — Returns only the paths of files containing at least one match.
- `"content"` — Returns matching lines in `file:line:content` format.
- `"count"` — Returns the number of matching lines per file in `file:count` format.

#### Special outputs

- Invalid regular expression: `"Invalid regex pattern: ..."`
- Invalid include pattern: `"Invalid include pattern"`
- No matching content: `"No matches found"`

When `use_ripgrep` is enabled, the tool first tries the `ripgrep` implementation. It uses the Python implementation when `ripgrep` is unavailable, disabled, or times out.

---

## Methods

### 1. `_validate_and_resolve_path`

Converts a virtual path into a resolved filesystem path and verifies that it remains inside `root_path`.

A leading `/` is added when the supplied path does not already contain one.

> This is an internal helper method.

- **Syntax:**

  ```python
  _validate_and_resolve_path(
      self,
      path: str # Virtual path relative to the configured root
  ) -> Path
  ```

- **Returns:** The resolved filesystem `Path`.
- **Raises:** `ValueError` when path traversal is detected or the resolved path lies outside `root_path`.

Paths containing `..` or `~` are rejected.

---

### 2. `_ripgrep_search`

Searches file contents using the external `ripgrep` command and parses its JSON output.

The command is executed with a 30-second timeout. When `ripgrep` is missing or times out, the method falls back to `_python_search`.

> This is an internal helper method.

- **Syntax:**

  ```python
  _ripgrep_search(
      self,
      pattern: str, # Regular expression to search for
      base_path: str, # Virtual path to search
      include: str | None # Optional filename glob filter
  ) -> dict[str, list[tuple[int, str]]]
  ```

- **Returns:** A dictionary mapping each matching virtual file path to a list of `(line_number, line_text)` tuples.
- Returns an empty dictionary when the path is invalid, does not exist, or no matches are found.

Results resolving outside `root_path` are discarded.

---

### 3. `_python_search`

Fallback content-search implementation using Python regular expressions and filesystem traversal.

It walks directories without following symlinked directories, ignores files outside the configured root, applies the optional include filter, and skips files larger than `max_file_size_bytes`.

> This is an internal helper method.

- **Syntax:**

  ```python
  _python_search(
      self,
      pattern: str, # Compiled-compatible regular expression
      base_path: str, # Virtual path to search
      include: str | None # Optional filename glob filter
  ) -> dict[str, list[tuple[int, str]]]
  ```

- **Returns:** A dictionary mapping each matching virtual file path to a list of `(line_number, line_text)` tuples.

Files that cannot be decoded as text or cannot be read because of permissions are skipped.

---

### 4. `_format_grep_results`

Formats the internal grep-result dictionary according to the selected output mode.

> This is an internal static helper method.

- **Syntax:**

  ```python
  _format_grep_results(
      results: dict[str, list[tuple[int, str]]], # Matches grouped by file
      output_mode: str # Requested result format
  ) -> str
  ```

- **Returns:** A newline-separated string formatted as file paths, matching content, or match counts.
- Unknown output modes default to the `"files_with_matches"` format.

---

## Internal Module Functions

### 1. `_is_within_root`

Checks whether a candidate path resolves inside an allowed root directory.

Both paths are resolved before comparison, so symbolic links and `..` segments are taken into account.

- **Syntax:**

  ```python
  _is_within_root(
      candidate: Path, # Path being checked
      root: Path # Allowed root directory
  ) -> bool
  ```

- **Returns:** `True` when the resolved candidate is inside the resolved root; otherwise `False`.
- Resolution errors also produce `False`.

---

### 2. `_expand_include_patterns`

Expands brace-style include patterns into individual glob patterns.

For example, `"*.{py,pyi}"` becomes `['*.py', '*.pyi']`.

- **Syntax:**

  ```python
  _expand_include_patterns(
      pattern: str # Include pattern that may contain braces
  ) -> list[str] | None
  ```

- **Returns:** A list of expanded glob patterns.
- Returns `None` when braces are malformed or empty.

Nested or repeated brace groups are expanded recursively.

---

### 3. `_is_valid_include_pattern`

Validates an include glob before it is passed to a search implementation.

The pattern must be non-empty, contain no null or newline characters, have valid brace syntax, and translate into compilable regular expressions.

- **Syntax:**

  ```python
  _is_valid_include_pattern(
      pattern: str # Include glob being validated
  ) -> bool
  ```

- **Returns:** `True` for a valid include pattern; otherwise `False`.

---

### 4. `_match_include_pattern`

Checks whether a filename matches an include pattern.

Brace patterns are expanded before matching, and matching is performed against the filename rather than the complete path.

- **Syntax:**

  ```python
  _match_include_pattern(
      basename: str, # Filename without its parent path
      pattern: str # Include glob pattern
  ) -> bool
  ```

- **Returns:** `True` when the filename matches at least one expanded pattern; otherwise `False`.

---

## Example

```python
from langchain.agents import create_agent
from langchain.agents.middleware import FilesystemFileSearchMiddleware

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        FilesystemFileSearchMiddleware(
            root_path="/workspace",
            use_ripgrep=True,
            max_file_size_mb=10,
        )
    ],
)
```

---

## Exported Components

```python
__all__ = [
    "FilesystemFileSearchMiddleware",
]
```